# NeuroBot — export of the deployment models (flagship TCFormer, ONNX)

Trains the TCFormer recipe of the paper for the two flagship subjects (Task A: S007, Task B: S002) with **one seed (seed 0)**,
then exports the fine-tuned **ME and MI specialists** to **ONNX** so the Render backend can serve them without PyTorch.

**v3 - two variants, chosen by `EXPORT_VARIANT` in the EXPORT cell:**

| variant | what it trains | time |
|---|---|---|
| `"baseline"` (default) | the subject's own earlier trials only, no other subjects | **about 30-60 min for both tasks** |
| `"cross"` | additionally pretrains on 30 other subjects first | **about 5-6 h per task** (run one task per session) |

The v1 run spent about 11 h on cross-subject pretraining for both tasks, hit the 11 h guard and exported nothing, because the pretrained weights were never saved and
the export came after the pretraining of *both* tasks. v3 fixes that: pretrained weights are saved the moment they exist, cached data can be restored from an earlier run, and the final cell
no longer fails when nothing was exported.

* Preprocessing is identical to the deployed backend (common average reference, 4-40 Hz FIR, 1 s windows, per-window z-score).
* Every export is checked: the ONNX output must equal the PyTorch output (max abs difference < 1e-3) or the cell stops with an error.
* The held-out accuracy of each exported model is written into its manifest entry, so the web app shows what the model really scored.
* **v2:** also writes a replay log per model (`replay_<task>_<mode>.json`: the exported model's predictions on its own held-out trials, 6 EEG channels per window) so the web app's replay shows the deployed models, not an older run.
* Output: `/kaggle/working/export/` (ONNX files, replay logs and `manifest_entries.json`) and `/kaggle/working/neurobot_models.zip`.

**Run:** GPU and Internet on, **Save Version -> Save & Run All**. With the default `"baseline"` variant expect about 30-60 min.
After it finishes, download `neurobot_models.zip` from the Output tab and follow the README of the project (section *Adding trained models*).

**Honesty note.** The flagship subjects were chosen earlier because they decode well, so their scores are best-case (as stated in the paper). The deployed models are
personal models of those subjects: they are meant to be used with those subjects' recordings (or the replay), not on arbitrary people.

---

The rest of this notebook is the v5.1 training notebook, unchanged except for the cells marked *EXPORT*.

In [ ]:
import os, glob, json, warnings, copy
warnings.filterwarnings("ignore")

import numpy as np
import mne
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

mne.set_log_level("ERROR")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "no GPU found - check accelerator setting")

## 1. Dataset, run mapping, preprocessing (unchanged)

In [ ]:
import os
import time
import mne
from mne.datasets import eegbci

MNE_DATA_DIR = "/kaggle/working/mne_data"
os.makedirs(MNE_DATA_DIR, exist_ok=True)

def edf_path(subject_id, run, max_retries=4, backoff_seconds=10):
    last_error = None
    for attempt in range(max_retries):
        try:
            paths = eegbci.load_data(subject_id, [run], path=MNE_DATA_DIR, update_path=False, verbose=False)
            return paths[0]
        except Exception as e:
            last_error = e
            if attempt < max_retries - 1:
                wait = backoff_seconds * (attempt + 1)
                print(f"    S{subject_id:03d} run {run}: download failed ({type(e).__name__}), retrying in {wait}s...")
                time.sleep(wait)
    raise last_error

test_path = edf_path(1, 1)
print("Download succeeded:", test_path)
print("File exists:", os.path.exists(test_path))

In [ ]:
LEFT_RIGHT_FIST_RUNS = {"ME": [3, 7, 11], "MI": [4, 8, 12]}
FISTS_FEET_RUNS = {"ME": [5, 9, 13], "MI": [6, 10, 14]}
LEFT_RIGHT_LABELS = {"T0": "rest", "T1": "left", "T2": "right"}
FISTS_FEET_LABELS = {"T0": "rest", "T1": "both_fists", "T2": "both_feet"}
KNOWN_BAD_SUBJECTS = [88, 92, 100, 104, 106]

In [ ]:
WINDOW_SEC, STEP_SEC = 1.0, 0.125
TRIAL_TMIN, TRIAL_TMAX = 0.0, 4.0

# Preprocessing ablations (defaults = the protocol behind every number reported so far). Set in the configuration cell.
NORM_MODE = "window"        # "window": z-score each channel inside every 1 s window (original)
                            # "run":    z-score each channel over the whole filtered run, so window-to-window power changes (ERD/ERS) survive
CHANNEL_SUBSET = None       # None = all 64 channels; or SENSORIMOTOR_CHANNELS (21 central channels, matched case-insensitively)
SENSORIMOTOR_CHANNELS = ["FC5", "FC3", "FC1", "FCz", "FC2", "FC4", "FC6", "C5", "C3", "C1", "Cz", "C2", "C4", "C6",
                         "CP5", "CP3", "CP1", "CPz", "CP2", "CP4", "CP6"]

def pick_channels(raw):
    if CHANNEL_SUBSET is None:
        return raw
    wanted = {c.lower() for c in CHANNEL_SUBSET}
    keep = [ch for ch in raw.ch_names if ch.lower() in wanted]
    if len(keep) != len(wanted):
        raise ValueError(f"channel subset: found {len(keep)} of {len(wanted)} requested channels")
    raw.pick(keep)
    return raw

def load_and_preprocess(path):
    raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
    raw.rename_channels(lambda ch: ch.strip("."))
    raw.set_eeg_reference("average", projection=False, verbose=False)
    raw.filter(l_freq=4.0, h_freq=40.0, fir_design="firwin", verbose=False)
    raw = pick_channels(raw)                 # after the common average reference, which uses all 64 channels
    return raw

def epoch_run(raw, run_number, label_map):
    events, event_id = mne.events_from_annotations(raw, verbose=False)
    class_event_id = {label_map[c]: eid for c, eid in event_id.items() if c in label_map}
    return mne.Epochs(raw, events, event_id=class_event_id, tmin=TRIAL_TMIN, tmax=TRIAL_TMAX,
                       baseline=None, preload=True, verbose=False)

def sliding_windows(epochs, sfreq, ref_stats=None):
    data = epochs.get_data()
    labels = epochs.events[:, -1]
    id_to_label = {v: k for k, v in epochs.event_id.items()}
    win_n, step_n = int(WINDOW_SEC * sfreq), int(STEP_SEC * sfreq)
    X, y, trial_ids = [], [], []
    for i in range(data.shape[0]):
        trial, label = data[i], id_to_label[labels[i]]
        start = 0
        while start + win_n <= trial.shape[1]:
            w = trial[:, start:start + win_n]
            if ref_stats is None:
                w = (w - w.mean(axis=1, keepdims=True)) / (w.std(axis=1, keepdims=True) + 1e-8)      # per-window (original)
            else:
                w = (w - ref_stats[0]) / (ref_stats[1] + 1e-8)                                         # per-run statistics
            X.append(w); y.append(label); trial_ids.append(i)
            start += step_n
    return np.stack(X), np.array(y), np.array(trial_ids)

def load_subjects(subject_ids, mode_runs, label_map, exclude_bad=True):
    if exclude_bad:
        subject_ids = [s for s in subject_ids if s not in KNOWN_BAD_SUBJECTS]
    Xs, ys, trial_id_chunks, mode_chunks = [], [], [], []
    trial_offset = 0
    for sid in subject_ids:
        for mode, runs in mode_runs.items():
            for run in runs:
                path = edf_path(sid, run)
                if not os.path.exists(path):
                    continue
                try:
                    raw = load_and_preprocess(path)
                    epochs = epoch_run(raw, run, label_map)
                    stats_run = None
                    if NORM_MODE == "run":
                        d = raw.get_data(); stats_run = (d.mean(axis=1, keepdims=True), d.std(axis=1, keepdims=True))
                    X, y, tids = sliding_windows(epochs, raw.info["sfreq"], stats_run)
                except Exception as e:
                    print(f"  skipping S{sid:03d} run {run}: {e}")
                    continue
                Xs.append(X); ys.append(y)
                trial_id_chunks.append(tids + trial_offset)
                mode_chunks.append(np.full(len(y), mode))
                trial_offset += tids.max() + 1 if len(tids) else 0
    return np.concatenate(Xs), np.concatenate(ys), np.concatenate(trial_id_chunks), np.concatenate(mode_chunks)

## 2. Training helpers (unchanged)

In [ ]:
def make_loader(X, y, batch_size=16, shuffle=True):
    ds = torch.utils.data.TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    return torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

def class_weights(y_int, n_classes):
    counts = np.bincount(y_int, minlength=n_classes).astype(np.float32)
    weights = counts.sum() / (n_classes * np.maximum(counts, 1))
    return torch.tensor(weights, dtype=torch.float32)

def augment_batch(xb, noise_std=0.05, max_shift=8):
    noise = torch.randn_like(xb) * noise_std
    xb = xb + noise
    shift = np.random.randint(-max_shift, max_shift + 1)
    if shift != 0:
        xb = torch.roll(xb, shifts=shift, dims=-1)
    return xb


class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_steps, total_steps, base_lr):
        self.optimizer = optimizer
        self.warmup_steps = max(1, warmup_steps)
        self.total_steps = max(total_steps, self.warmup_steps + 1)
        self.base_lr = base_lr
        self.step_count = 0

    def step_batch(self):
        self.step_count += 1
        if self.step_count <= self.warmup_steps:
            lr = self.base_lr * self.step_count / self.warmup_steps
        else:
            progress = (self.step_count - self.warmup_steps) / (self.total_steps - self.warmup_steps)
            progress = min(progress, 1.0)
            lr = 0.5 * self.base_lr * (1 + np.cos(np.pi * progress))
        for g in self.optimizer.param_groups:
            g["lr"] = lr

    def step_epoch(self):
        pass


def train_loop(model, loader, val_loader, optimizer, criterion, max_epochs, patience=20,
                verbose=True, augment=True, scheduler=None):
    own_scheduler = scheduler is None
    if own_scheduler:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=8)
    best_val_loss = float('inf')
    best_state = None
    best_epoch = 0
    epochs_without_improvement = 0
    history = []

    for epoch in range(max_epochs):
        model.train()
        total_loss, total_correct, total_n = 0.0, 0, 0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            if augment:
                xb = augment_batch(xb)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward(); optimizer.step()
            if scheduler is not None and hasattr(scheduler, "step_batch"):
                scheduler.step_batch()
            total_loss += loss.item() * xb.size(0)
            total_correct += (logits.argmax(1) == yb).sum().item()
            total_n += xb.size(0)
        train_loss = total_loss / total_n
        train_acc = total_correct / total_n

        model.eval()
        val_loss, val_correct, val_n = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits = model(xb)
                val_loss += criterion(logits, yb).item() * xb.size(0)
                val_correct += (logits.argmax(1) == yb).sum().item()
                val_n += xb.size(0)
        val_loss /= val_n
        val_acc = val_correct / val_n

        if own_scheduler:
            scheduler.step(val_loss)
        elif hasattr(scheduler, "step_epoch"):
            scheduler.step_epoch()

        history.append({"epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss,
                         "train_acc": train_acc, "val_acc": val_acc,
                         "lr": optimizer.param_groups[0]["lr"]})

        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            best_epoch = epoch + 1
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if verbose and ((epoch + 1) % 10 == 0 or epoch == 0):
            print(f"    epoch {epoch+1}/{max_epochs}  train_loss={train_loss:.4f} train_acc={train_acc:.3f}  "
                  f"val_loss={val_loss:.4f} val_acc={val_acc:.3f}  lr={optimizer.param_groups[0]['lr']:.1e}")

        if epochs_without_improvement >= patience:
            if verbose:
                print(f"    early stopping at epoch {epoch+1} (no val improvement for {patience} epochs); "
                      f"best epoch was {best_epoch} (val_loss={best_val_loss:.4f})")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history


@torch.no_grad()
def evaluate(model, loader, label_order):
    model.eval()
    preds, trues = [], []
    for xb, yb in loader:
        p = model(xb.to(DEVICE)).argmax(1).cpu().numpy()
        preds.extend(p); trues.extend(yb.numpy())
    acc = accuracy_score(trues, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(trues, preds, average=None,
                                                          labels=range(len(label_order)), zero_division=0)
    cm = confusion_matrix(trues, preds, labels=range(len(label_order)))
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "confusion_matrix": cm}

@torch.no_grad()
def evaluate_majority_vote(model, X, y_int, trial_ids, label_order):
    model.eval()
    loader = make_loader(X, y_int, shuffle=False)
    window_preds = []
    for xb, _ in loader:
        window_preds.extend(model(xb.to(DEVICE)).argmax(1).cpu().numpy())
    window_preds = np.array(window_preds)

    trial_true, trial_pred = [], []
    for t in np.unique(trial_ids):
        mask = trial_ids == t
        trial_true.append(y_int[mask][0])
        trial_pred.append(np.bincount(window_preds[mask], minlength=len(label_order)).argmax())
    acc = accuracy_score(trial_true, trial_pred)
    cm = confusion_matrix(trial_true, trial_pred, labels=range(len(label_order)))
    return {"accuracy": acc, "n_trials": len(trial_true), "confusion_matrix": cm}


def plot_training_curve(history, title):
    epochs = [h["epoch"] for h in history]
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
    axes[0].plot(epochs, [h["train_loss"] for h in history], label="train")
    axes[0].plot(epochs, [h["val_loss"] for h in history], label="val")
    axes[0].set_title(f"{title} - loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
    axes[1].plot(epochs, [h["train_acc"] for h in history], label="train")
    axes[1].plot(epochs, [h["val_acc"] for h in history], label="val")
    axes[1].set_ylim(0, 1); axes[1].set_title(f"{title} - accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
    axes[2].plot(epochs, [h["lr"] for h in history], color="#888888")
    axes[2].set_title(f"{title} - learning rate"); axes[2].set_xlabel("epoch")
    plt.tight_layout(); plt.show()
    best_epoch = max(range(len(history)), key=lambda i: history[i]["val_acc"])
    print(f"  Ran {len(history)} epochs. Best val accuracy {history[best_epoch]['val_acc']:.3f} "
          f"at epoch {history[best_epoch]['epoch']} (final epoch val_acc={history[-1]['val_acc']:.3f}).")

## 3. EEGNet (the CNN) — same architecture and settings as v2

In [ ]:
class EEGNet(nn.Module):
    def __init__(self, n_channels, n_samples, n_classes, F1=8, D=2, F2=16, kernel_length=32, dropout=0.5):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, (1, kernel_length), padding=(0, kernel_length // 2), bias=False),
            nn.BatchNorm2d(F1),
            nn.Conv2d(F1, F1 * D, (n_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D), nn.ELU(),
            nn.AvgPool2d((1, 4)), nn.Dropout(dropout),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(F1 * D, F1 * D, (1, 16), padding=(0, 8), groups=F1 * D, bias=False),
            nn.Conv2d(F1 * D, F2, (1, 1), bias=False),
            nn.BatchNorm2d(F2), nn.ELU(),
            nn.AvgPool2d((1, 8)), nn.Dropout(dropout),
        )
        self.eval()                               # size probe in eval mode: must not touch BatchNorm statistics
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, n_samples)
            flat_dim = self.block2(self.block1(dummy)).numel()
        self.train()
        self.classifier = nn.Linear(flat_dim, n_classes)

    def forward(self, x):
        if x.dim() == 3:
            x = x.unsqueeze(1)
        x = self.block1(x); x = self.block2(x)
        return self.classifier(x.flatten(1))

    def set_dropout(self, p):
        for m in self.modules():
            if isinstance(m, nn.Dropout):
                m.p = p

def eegnet_factory(n_channels, n_samples, n_classes):
    return EEGNet(n_channels, n_samples, n_classes, dropout=0.5)

## 4. Protocol — v3 trial-level splits, `seed`, `init_state` (cross-subject initialisation)

In [ ]:
import random

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def freeze_early_layers(model, keep_frac=0.3):
    params = list(model.named_parameters())
    cutoff = int(len(params) * (1 - keep_frac))
    for i, (name, p) in enumerate(params):
        p.requires_grad = (i >= cutoff)
    return [p for p in model.parameters() if p.requires_grad]


def make_optimizer(params, name, lr, weight_decay):
    if name == "adamw":
        return torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    return torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)


SPLIT_MODE = "trial"      # "trial" = correct (no leakage); "window" = legacy v2 behaviour (leaky)
BASE_FRAC, FT_FRAC = 0.60, 0.20   # remaining 0.20 of trials = held-out evaluation


def split_by_trial(trial_ids, y_int, test_size, seed=0):
    """Boolean masks (train_mask, test_mask) such that every trial goes wholly to one side."""
    uniq = np.unique(trial_ids)
    lab = np.array([y_int[trial_ids == t][0] for t in uniq])
    try:
        tr, te = train_test_split(uniq, test_size=test_size, stratify=lab, random_state=seed)
    except ValueError:    # a class has too few trials to stratify
        tr, te = train_test_split(uniq, test_size=test_size, random_state=seed)
    return np.isin(trial_ids, tr), np.isin(trial_ids, te)


def plan_splits(y_int, trial_ids, modes, split_mode=None, seed=0):
    """Returns (base_train_idx, base_val_idx, {mode: {ft_train, ft_val, eval}}) as index arrays."""
    split_mode = split_mode or SPLIT_MODE
    base_parts, late = [], {}
    for mode in np.unique(modes):
        idx = np.where(modes == mode)[0]
        if split_mode == "trial":
            ordered = np.unique(trial_ids[idx])                  # ids grow with recording order
            n_base = max(1, int(round(len(ordered) * BASE_FRAC)))
            in_base = np.isin(trial_ids[idx], ordered[:n_base])
            base_idx, late_idx = idx[in_base], idx[~in_base]
        else:                                                     # legacy: first 70 % of windows
            cut = int(len(idx) * 0.7)
            base_idx, late_idx = idx[:cut], idx[cut:]
        base_parts.append(base_idx); late[mode] = late_idx
    base_idx = np.concatenate(base_parts)

    if split_mode == "trial":
        tr_m, va_m = split_by_trial(trial_ids[base_idx], y_int[base_idx], 0.15, seed)
        base_train, base_val = base_idx[tr_m], base_idx[va_m]
    else:
        shuffled = base_idx[np.random.RandomState(0).permutation(len(base_idx))]
        cut = int(len(shuffled) * 0.85)
        base_train, base_val = shuffled[:cut], shuffled[cut:]

    plans = {}
    for mode, late_idx in late.items():
        if len(late_idx) < 20:
            continue
        if split_mode == "trial":
            eval_share = (1 - BASE_FRAC - FT_FRAC) / (1 - BASE_FRAC)
            ft_m, ev_m = split_by_trial(trial_ids[late_idx], y_int[late_idx], eval_share, seed)
            ft_idx, eval_idx = late_idx[ft_m], late_idx[ev_m]
            a, b = split_by_trial(trial_ids[ft_idx], y_int[ft_idx], 0.2, seed)
            ft_train, ft_val = ft_idx[a], ft_idx[b]
        else:
            ft_idx, eval_idx = train_test_split(late_idx, test_size=0.5, stratify=y_int[late_idx], random_state=0)
            ft_train, ft_val = train_test_split(ft_idx, test_size=0.2, stratify=y_int[ft_idx], random_state=0)
        plans[mode] = {"ft_train": ft_train, "ft_val": ft_val, "eval": eval_idx}
    return base_train, base_val, plans


# ---- self-test (synthetic, no GPU/data needed): trial mode must never share a trial across parts ----
def _selftest_splits():
    rng = np.random.RandomState(1)
    n_per_mode, wins = 60, 25
    trial_ids, y, modes = [], [], []
    for m_i, m in enumerate(["ME", "MI"]):
        for t in range(n_per_mode):
            tid = m_i * n_per_mode + t
            lab = rng.randint(0, 3)
            trial_ids += [tid] * wins; y += [lab] * wins; modes += [m] * wins
    trial_ids, y, modes = np.array(trial_ids), np.array(y), np.array(modes)
    for sm, expect_clean in [("trial", True), ("window", False)]:
        btr, bva, plans = plan_splits(y, trial_ids, modes, split_mode=sm)
        T = lambda idx: set(trial_ids[idx].tolist())
        clean = not (T(btr) & T(bva))
        for p in plans.values():
            clean &= not (T(p["ft_train"]) & T(p["ft_val"])) and not (T(p["ft_train"]) & T(p["eval"])) and not (T(p["ft_val"]) & T(p["eval"]))
            clean &= not (T(btr) & T(p["eval"]))
        assert clean == expect_clean, f"split self-test failed for {sm}"
    print("split self-test passed: trial mode shares no trial across parts; window mode (legacy) does.")
_selftest_splits()


def _finetune_on_subset(base_model, X_ft_train, y_ft_train, X_ft_val, y_ft_val, X_eval, y_eval, trial_eval, label_order,
                         ft_epochs, n_classes, optimizer_name="adam", ft_weight_decay=0.0, label_smoothing=0.0,
                         keep_frac=0.3, use_warmup=False, warmup_frac=0.1, ft_patience=12, ft_lr=5e-4):
    model = copy.deepcopy(base_model)
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.p = 0.65
    trainable = freeze_early_layers(model, keep_frac=keep_frac)
    weights = class_weights(y_ft_train, n_classes).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=label_smoothing)
    ft_opt = make_optimizer(trainable, optimizer_name, lr=ft_lr, weight_decay=ft_weight_decay)
    ft_loader = make_loader(X_ft_train, y_ft_train)
    scheduler = None
    if use_warmup:
        steps_per_epoch = max(1, len(ft_loader))
        scheduler = WarmupCosineScheduler(ft_opt, warmup_steps=int(steps_per_epoch * warmup_frac * ft_epochs) or steps_per_epoch,
                                           total_steps=steps_per_epoch * ft_epochs, base_lr=ft_lr)
    model, history = train_loop(model, ft_loader, make_loader(X_ft_val, y_ft_val, shuffle=False),
                                 ft_opt, criterion, max_epochs=ft_epochs, patience=ft_patience, verbose=False,
                                 scheduler=scheduler)
    ft_m = evaluate(model, make_loader(X_eval, y_eval, shuffle=False), label_order)
    mv_m = evaluate_majority_vote(model, X_eval, y_eval, trial_eval, label_order)
    return model, ft_m, mv_m, (X_eval, y_eval, trial_eval), history


def run_finetune_protocol_hybrid(X, y_int, trial_ids, modes, label_order, model_fn,
                                  base_epochs=150, ft_epochs=60, base_patience=20, ft_patience=12,
                                  verbose=True, optimizer_name="adam", weight_decay=0.0, label_smoothing=0.0,
                                  keep_frac=0.3, use_warmup=False, warmup_frac=0.1, base_lr=1e-3, ft_lr=5e-4,
                                  init_state=None, seed=None):
    if seed is not None:
        set_seed(seed)
    n_channels, n_samples = X.shape[1], X.shape[2]
    n_classes = len(label_order)
    base_train, base_val, plans = plan_splits(y_int, trial_ids, modes)
    X_train, y_train, X_val, y_val = X[base_train], y_int[base_train], X[base_val], y_int[base_val]

    base_model = model_fn(n_channels, n_samples, n_classes).to(DEVICE)
    if init_state is not None:               # cross-subject pretrained weights
        base_model.load_state_dict(init_state)
    weights = class_weights(y_train, n_classes).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=label_smoothing)
    if verbose:
        print("  Pretraining base model on POOLED ME+MI data...")
    opt = make_optimizer(base_model.parameters(), optimizer_name, lr=base_lr, weight_decay=weight_decay)
    base_loader = make_loader(X_train, y_train)
    scheduler = None
    if use_warmup:
        steps_per_epoch = max(1, len(base_loader))
        scheduler = WarmupCosineScheduler(opt, warmup_steps=int(steps_per_epoch * warmup_frac * base_epochs) or steps_per_epoch,
                                           total_steps=steps_per_epoch * base_epochs, base_lr=base_lr)
    base_model, base_history = train_loop(base_model, base_loader, make_loader(X_val, y_val, shuffle=False),
                                           opt, criterion, max_epochs=base_epochs, patience=base_patience, verbose=verbose,
                                           scheduler=scheduler)

    results_by_mode = {}
    for mode, p in plans.items():
        if len(p["eval"]) == 0 or len(p["ft_val"]) == 0:
            if verbose:
                print(f"  {mode}: skipped (too few trials for a fine-tune/validation/evaluation split)")
            continue
        if verbose:
            print(f"  Fine-tuning on {mode}-only data...")
        model, ft_m, mv_m, eval_data, ft_history = _finetune_on_subset(
            base_model, X[p["ft_train"]], y_int[p["ft_train"]], X[p["ft_val"]], y_int[p["ft_val"]],
            X[p["eval"]], y_int[p["eval"]], trial_ids[p["eval"]], label_order, ft_epochs, n_classes,
            optimizer_name=optimizer_name, ft_weight_decay=weight_decay, label_smoothing=label_smoothing,
            keep_frac=keep_frac, use_warmup=use_warmup, warmup_frac=warmup_frac, ft_patience=ft_patience, ft_lr=ft_lr)
        if verbose:
            print(f"    {mode} - per-window: {ft_m['accuracy']:.3f}  |  majority-vote ({mv_m['n_trials']} trials): {mv_m['accuracy']:.3f}")
        results_by_mode[mode] = {"model": model, "finetuned": ft_m["accuracy"],
                                  "majority_vote": mv_m["accuracy"], "n_trials": mv_m["n_trials"],
                                  "eval_data": eval_data, "history": ft_history}
    return base_model, results_by_mode, base_history


def run_per_subject_evaluation_hybrid(subject_ids, mode_runs, label_map, label_order, model_fn,
                                       base_epochs=150, ft_epochs=60, progress_tag=None, **kwargs):
    subject_ids = [s for s in subject_ids if s not in KNOWN_BAD_SUBJECTS]
    all_results = {"ME": [], "MI": []}
    for sid in subject_ids:
        try:
            X, y, trial_ids, modes = load_subjects([sid], mode_runs, label_map, exclude_bad=False)
        except Exception as e:
            print(f"  S{sid:03d}: skipped (data loading failed after retries: {type(e).__name__}: {e})")
            continue
        if len(X) < 60:
            print(f"  S{sid:03d}: skipped (too little data: {len(X)} windows)")
            continue
        y_int = np.array([label_order.index(l) for l in y])
        try:
            base_model, results_by_mode, base_history = run_finetune_protocol_hybrid(
                X, y_int, trial_ids, modes, label_order, model_fn,
                base_epochs=base_epochs, ft_epochs=ft_epochs, verbose=False, **kwargs)
        except Exception as e:
            print(f"  S{sid:03d}: training failed ({e})")
            continue
        line = f"  S{sid:03d}:"
        for mode in ["ME", "MI"]:
            if mode in results_by_mode:
                r = results_by_mode[mode]
                line += f"  {mode} majority-vote={r['majority_vote']:.3f} ({r['n_trials']} trials, {len(r['history'])} ft epochs)"
                all_results[mode].append({"subject": sid, "majority_vote": r["majority_vote"],
                                           "n_trials": r["n_trials"], "model": r["model"],
                                           "eval_data": r["eval_data"], "n_channels": X.shape[1], "n_samples": X.shape[2],
                                           "base_history": base_history, "history": r["history"]})
        print(line)
        if progress_tag:                      # survive a stopped/disconnected session
            rec = {"subject": int(sid)}
            for m in results_by_mode:
                rec[m] = {"majority_vote": float(results_by_mode[m]["majority_vote"]),
                          "n_trials": int(results_by_mode[m]["n_trials"])}
            with open(f"/kaggle/working/progress_{progress_tag}.jsonl", "a") as fh:
                fh.write(json.dumps(rec) + "\n")
    return all_results

def summarize_results(results, task_name):
    mv = [r["majority_vote"] for r in results]
    print(f"\n{task_name}: {len(results)} subjects evaluated "
          f"(mean {np.mean([r['n_trials'] for r in results]):.1f} held-out trials per subject)")
    print(f"  Majority-vote accuracy - mean: {np.mean(mv):.3f}  std: {np.std(mv):.3f}  "
          f"min: {np.min(mv):.3f}  max: {np.max(mv):.3f}")
    return np.mean(mv), np.std(mv)

def plot_per_subject(results, label_order, title, color):
    fig, ax = plt.subplots(figsize=(8, 4))
    subs = [r["subject"] for r in results]
    mv_vals = [r["majority_vote"] for r in results]
    ax.bar([str(s) for s in subs], mv_vals, color=color)
    ax.axhline(np.mean(mv_vals), color="black", linestyle="--", label=f"mean={np.mean(mv_vals):.2f}")
    ax.axhline(1/len(label_order), color="red", linestyle=":", label=f"chance ({1/len(label_order):.2f})")
    ax.set_ylim(0, 1); ax.set_ylabel("Majority-vote accuracy"); ax.set_xlabel("Subject")
    ax.set_title(title); ax.legend()
    plt.tight_layout(); plt.show()

def pick_result(results, preferred_subject=None):
    if preferred_subject is not None:
        for r in results:
            if r["subject"] == preferred_subject:
                return r
    return max(results, key=lambda r: r["majority_vote"])


def summarize_progress(tag):
    """Re-computes the group tables from the progress file (works even if the training cell never finished)."""
    path = f"/kaggle/working/progress_{tag}.jsonl"
    if not os.path.exists(path):
        print(f"{tag}: no progress file yet ({path})"); return
    rows = [json.loads(l) for l in open(path) if l.strip()]
    print(f"\n{tag}: {len(rows)} subject(s) recorded")
    for mode in ["ME", "MI"]:
        vals = [r[mode]["majority_vote"] for r in rows if mode in r]
        if vals:
            print(f"  {mode}: mean {np.mean(vals):.3f}  std {np.std(vals):.3f}  min {np.min(vals):.3f}  max {np.max(vals):.3f}  "
                  f"(n={len(vals)})  per-subject: " + ", ".join(f"S{r['subject']:03d}={r[mode]['majority_vote']:.3f}" for r in rows if mode in r))

## 5. Leakage and consistency checks

`verify_no_leakage` re-derives the split for a subject and **raises an error** if any trial id appears in two of
{pretrain, pretrain-validation, fine-tune, fine-tune-validation, evaluation}, or if an evaluation window belongs to
the wrong mode. It runs before every subject. `check_probs` rejects NaNs/Infs and wrong shapes.

In [ ]:
def verify_no_leakage(y_int, trial_ids, modes, label):
    base_train, base_val, plans = plan_splits(y_int, trial_ids, modes)
    T = lambda idx: set(trial_ids[idx].tolist())
    assert len(plans) > 0, f"{label}: no mode has enough trials for a fine-tune/evaluation split"
    assert not (T(base_train) & T(base_val)), f"{label}: pretrain and pretrain-validation share a trial"
    for mode, p in plans.items():
        parts = {"pretrain": T(base_train), "pretrain_val": T(base_val), "ft_train": T(p["ft_train"]),
                 "ft_val": T(p["ft_val"]), "eval": T(p["eval"])}
        names = list(parts)
        for i in range(len(names)):
            for j in range(i + 1, len(names)):
                shared = parts[names[i]] & parts[names[j]]
                assert not shared, f"{label} {mode}: LEAKAGE - {len(shared)} trial(s) in both {names[i]} and {names[j]}"
        assert (modes[p["eval"]] == mode).all(), f"{label} {mode}: evaluation windows from another mode"
        assert len(parts["eval"]) >= 3, f"{label} {mode}: fewer than 3 evaluation trials"
    return {m: len(T(p["eval"])) for m, p in plans.items()}

def check_probs(p, n_rows, n_classes, label):
    assert p.shape == (n_rows, n_classes), f"{label}: probability shape {p.shape}, expected {(n_rows, n_classes)}"
    assert np.isfinite(p).all(), f"{label}: NaN/Inf in predicted probabilities"
    assert np.allclose(p.sum(1), 1.0, atol=1e-3), f"{label}: probabilities do not sum to 1"

# self-test on synthetic data (no GPU needed): the real split must pass, the legacy window split must be caught
def _selftest_verify():
    rng = np.random.RandomState(3); wins = 25
    tid, y, modes = [], [], []
    for mi, m in enumerate(["ME", "MI"]):
        for t in range(90):
            tid += [mi * 90 + t] * wins; y += [rng.randint(0, 3)] * wins; modes += [m] * wins
    tid, y, modes = np.array(tid), np.array(y), np.array(modes)
    verify_no_leakage(y, tid, modes, "selftest")
    global SPLIT_MODE
    old = SPLIT_MODE; SPLIT_MODE = "window"
    try:
        verify_no_leakage(y, tid, modes, "selftest-legacy"); caught = False
    except AssertionError:
        caught = True
    finally:
        SPLIT_MODE = old
    assert caught, "verify_no_leakage failed to detect the legacy window-level leak"
    print("leakage self-test passed: trial split is clean, legacy window split is detected.")
_selftest_verify()

## 6. Configuration

In [ ]:
DRY_RUN = False                      # True = 1 subject, tiny epochs, both models, both variants (error check only)
FAST_PILOT = False                   # True = v4 pilot: Task A, subjects 1-4, TCFormer, baseline vs cross, 2 seeds
RUN_TASKS = ["B"]                    # one task per session. Task A (cross, both models) is finished -> v5_results.json. Task B is next.
MODELS_TO_RUN = ["tcformer", "eegnet"]
VARIANTS = ["cross"]                 # add "baseline" to also run without cross-subject pretraining
SEEDS = {"tcformer": [0, 1, 2], "eegnet": [0, 1, 2]}
EVAL_SUBJECTS = list(range(1, 11))
N_EXT = 30                           # external pretraining subjects (ids 11 and up, never an evaluated subject)
EXT_EPOCHS, EXT_PATIENCE = 40, 6
EXT_WINDOW_STRIDE = 2                # keep every 2nd window of external subjects

LABEL_ORDER_LR = ["left", "rest", "right"]
LABEL_ORDER_FF = ["both_feet", "both_fists", "rest"]
TASK_CFG = {"A": dict(runs=LEFT_RIGHT_FIST_RUNS, labels=LEFT_RIGHT_LABELS, order=LABEL_ORDER_LR),
            "B": dict(runs=FISTS_FEET_RUNS, labels=FISTS_FEET_LABELS, order=LABEL_ORDER_FF)}
EPOCH_OVERRIDE = None

ABLATION_BASELINE = False            # True: TCFormer WITHOUT cross-subject pretraining, same held-out trials (measures the cross-subject gain)
NORM_MODE = "window"                 # preprocessing ablations, see section 1; keep the defaults for the primary result
CHANNEL_SUBSET = None                # None or SENSORIMOTOR_CHANNELS
MAX_RUNTIME_HOURS = 11.0             # Kaggle kills a session at 12 h and keeps NO output; stop starting new subjects before that
MIN_BLOCK_MINUTES = 75               # a subject needs roughly 40-60 min; do not start one with less than this left
PRIOR_RESULTS_PATHS = []             # e.g. ["/kaggle/input/my-v5-run/v5_results.json"]: merged in, finished subjects are skipped
T0 = time.time()
DATA_TAG = f"{NORM_MODE}_{'allch' if CHANNEL_SUBSET is None else str(len(CHANNEL_SUBSET)) + 'ch'}"
if ABLATION_BASELINE:
    MODELS_TO_RUN, VARIANTS = ["tcformer"], ["baseline"]
if FAST_PILOT:
    RUN_TASKS, EVAL_SUBJECTS, N_EXT = ["A"], [1, 2, 3, 4], 20
    MODELS_TO_RUN, VARIANTS, SEEDS = ["tcformer"], ["baseline", "cross"], {"tcformer": [0, 1], "eegnet": [0, 1]}
if DRY_RUN:
    RUN_TASKS, EVAL_SUBJECTS, N_EXT = ["A"], [1], 2
    MODELS_TO_RUN, VARIANTS, SEEDS = ["tcformer", "eegnet"], ["baseline", "cross"], {"tcformer": [0], "eegnet": [0]}
    EXT_EPOCHS, EXT_PATIENCE, EPOCH_OVERRIDE = 2, 2, 3

EXT_IDS = [s for s in range(11, 110) if s not in KNOWN_BAD_SUBJECTS][:N_EXT]
assert not set(EXT_IDS) & set(EVAL_SUBJECTS), "external pretraining subjects must never include evaluated subjects"
assert set(MODELS_TO_RUN) <= {"tcformer", "eegnet"} and set(VARIANTS) <= {"baseline", "cross"}
if DRY_RUN:
    RESULTS_PATH = "/kaggle/working/v5_results_DRYRUN.json"
elif DATA_TAG == "window_allch":
    RESULTS_PATH = "/kaggle/working/v5_results.json"              # the protocol behind all reported numbers
else:
    RESULTS_PATH = f"/kaggle/working/v5_results_{DATA_TAG}.json"  # non-default preprocessing never mixes with the main file
PROBS_DIR = "/kaggle/working/probs"
os.makedirs(PROBS_DIR, exist_ok=True)

def time_left():
    return MAX_RUNTIME_HOURS * 3600 - (time.time() - T0)

def _json_default(o):
    if hasattr(o, "item"): return o.item()
    if hasattr(o, "tolist"): return o.tolist()
    return str(o)

def load_json_safe(path):
    """{} for a missing, empty (0-byte) or corrupt file instead of crashing."""
    try:
        if os.path.exists(path) and os.path.getsize(path) > 0:
            with open(path) as fh:
                return json.load(fh)
    except Exception as e:
        print(f"  could not read {path} ({type(e).__name__}: {e}) - ignoring it")
    return {}

def load_results_merged():
    merged = {}
    for p in list(PRIOR_RESULTS_PATHS) + [RESULTS_PATH]:
        part = load_json_safe(p)
        if part: print(f"  loaded {len(part)} record(s) from {p}")
        merged.update(part)
    return merged

def save_results(res):
    """Atomic: temp file, flush to disk, rename. Never leaves a half-written or empty JSON behind."""
    tmp = RESULTS_PATH + ".tmp"
    with open(tmp, "w") as fh:
        json.dump(res, fh, indent=1, default=_json_default)
        fh.flush(); os.fsync(fh.fileno())
    os.replace(tmp, RESULTS_PATH)
print(f"mode={'DRY_RUN' if DRY_RUN else 'FAST_PILOT' if FAST_PILOT else 'FULL'} | tasks={RUN_TASKS} | models={MODELS_TO_RUN} | "
      f"variants={VARIANTS} | eval subjects={EVAL_SUBJECTS} | seeds={SEEDS} | external subjects={len(EXT_IDS)}")
print(f"preprocessing={DATA_TAG} | results file: {RESULTS_PATH}")

In [ ]:
# EXPORT: overrides applied after the configuration cell
EXPORT_MODE = True
EXPORT_SUBJECTS = {"A": [7], "B": [2]}        # flagship subjects named in the paper
EXPORT_MODELS = ["tcformer"]
RUN_TASKS = ["A", "B"]
MODELS_TO_RUN = ["tcformer"]
EXPORT_VARIANT = "baseline"   # "baseline": subject's own trials only (30-60 min for both tasks). "cross": also pretrain on 30 other subjects (5-6 h PER TASK)
assert EXPORT_VARIANT in ("baseline", "cross")
VARIANTS = [EXPORT_VARIANT]
if EXPORT_VARIANT == "cross":
    RUN_TASKS = ["B"]         # one task per session; set ["A"] for the second session
MIN_BLOCK_MINUTES = 20        # a single exported subject takes minutes, not an hour
SEEDS = {"tcformer": [0], "eegnet": [0]}       # one seed: one model per mode is exported (no seed picking)
if DRY_RUN:
    EXPORT_SUBJECTS = {"A": [1], "B": [1]}
RESULTS_PATH = "/kaggle/working/export_run_results.json" if not DRY_RUN else "/kaggle/working/export_run_DRYRUN.json"
PRIOR_RESULTS_PATHS = []
USE_EMBEDDED = False
print("EXPORT_VARIANT:", EXPORT_VARIANT, "| tasks", RUN_TASKS)
print("EXPORT_MODE:", EXPORT_SUBJECTS, "| models", MODELS_TO_RUN, "| variants", VARIANTS, "| seeds", SEEDS, "| results", RESULTS_PATH)

In [ ]:
# EXPORT: ONNX export + verification helpers
import copy, zipfile
EXPORT_DIR = "/kaggle/working/export"; os.makedirs(EXPORT_DIR, exist_ok=True)
MANIFEST_ENTRIES = []

def _install_onnx():
    try:
        import onnx, onnxruntime  # noqa
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime"])
_install_onnx()
import onnxruntime as ort


REPLAY_CHANNELS = [8, 10, 12, 1, 5, 17]            # C3, Cz, C4, FC3, FC4, CPz in the PhysioNet 64-channel order
REPLAY_CHANNEL_NAMES = ["C3", "CZ", "C4", "FC3", "FC4", "CPZ"]

def write_replay_log(task, sid, mode, model_name, r, order, mid, accuracy_entry):
    """EXPORT (v2): replay log of the exported model on its own held-out trials, for the web app.
    Windows are listed trial by trial; `t` is replay time (125 ms per record), `trial` is the held-out trial id."""
    Xe, ye, te = r["eval_data"]
    p = predict_proba(r["model"], Xe)
    idx = np.argsort(np.asarray(te), kind="stable")          # keep each trial's windows together, in recording order
    records = []
    for j, i in enumerate(idx):
        records.append({
            "t": round(j * 0.125, 3), "trial": int(te[i]),
            "true_class": order[int(ye[i])], "predicted_class": order[int(np.argmax(p[i]))],
            "probs": {order[k]: float(p[i, k]) for k in range(len(order))},
            "waveform": [[round(float(v), 3) for v in Xe[i, c, ::4]] for c in REPLAY_CHANNELS]})
    per_window = float(np.mean([rec["true_class"] == rec["predicted_class"] for rec in records]))
    task_name = {"A": "left_right", "B": "fists_feet"}[task]
    doc = {"label_order": list(order), "channels_shown": REPLAY_CHANNELS, "channel_names": REPLAY_CHANNEL_NAMES,
           "meta": {"source": f"Held-out trials of the exported model {mid} (not training data)",
                    "model_id": mid, "subject": f"S{sid:03d}", "mode": mode, "seed": 0,
                    "majority_vote_accuracy": accuracy_entry["majority_vote_accuracy"],
                    "n_heldout_trials": accuracy_entry["n_heldout_trials"], "per_window_accuracy": per_window,
                    "n_windows": len(records),
                    "caveats": ["Best-case flagship subject chosen on this evaluation data; optimistic.",
                                "Time axis is replay time, not recording time; windows are grouped by trial."]},
           "records": records}
    path = os.path.join(EXPORT_DIR, f"replay_{task_name}_{mode}.json")
    with open(path, "w") as fh:
        json.dump(doc, fh)
    print(f"  replay log {os.path.basename(path)}: {len(records)} windows from {len(set(int(t) for t in te))} held-out trials, per-window {per_window:.3f}")

def export_entry(task, sid, mode, model_name, r, order, n_channels, n_samples):
    """Export one fine-tuned model to ONNX, verify it against PyTorch and record a manifest entry."""
    model = copy.deepcopy(r["model"]).cpu().eval()
    mid = f"{model_name}-{task}-s{sid:03d}-{mode.lower()}"
    path = os.path.join(EXPORT_DIR, mid + ".onnx")
    dummy = torch.randn(4, n_channels, n_samples)
    kw = dict(input_names=["window"], output_names=["logits"], dynamic_axes={"window": {0: "batch"}, "logits": {0: "batch"}}, opset_version=17)
    try:
        torch.onnx.export(model, dummy, path, dynamo=False, **kw)
    except TypeError:
        torch.onnx.export(model, dummy, path, **kw)
    sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    with torch.no_grad():
        ref = model(dummy).numpy()
    out = sess.run(None, {"window": dummy.numpy()})[0]
    err = float(np.abs(ref - out).max())
    # also check a different batch size (the server sends batches of up to 128 windows)
    big = torch.randn(37, n_channels, n_samples)
    with torch.no_grad():
        err2 = float(np.abs(model(big).numpy() - sess.run(None, {"window": big.numpy()})[0]).max())
    assert max(err, err2) < 1e-3, f"ONNX output differs from PyTorch by {max(err, err2):.2e} for {mid}; do not deploy this file"
    cfg = TASK_CFG[task]
    MANIFEST_ENTRIES.append({
        "id": mid, "task": {"A": "left_right", "B": "fists_feet"}[task], "mode": mode, "arch": model_name,
        "title": f"{model_name.upper() if model_name == 'eegnet' else 'TCFormer'}, subject S{sid:03d}, {'executed' if mode == 'ME' else 'imagined'} movement",
        "label_order": list(order), "onnx_path": mid + ".onnx", "output": "logits",
        "trained_on": f"PhysioNet eegmmidb subject S{sid:03d} ({mode} specialist), " + ("pretrained on 30 other subjects first" if EXPORT_VARIANT == "cross" else "trained on this subject's own earlier trials only (no cross-subject pretraining)") + ", seed 0",
        "variant": EXPORT_VARIANT,
        "protocol": "trial-level split (60% pretrain / 20% fine-tune / 20% held-out), as in the paper",
        "warning": f"Personal model of subject S{sid:03d}: meaningful only for that subject's recordings (or the replay), not for other people. Best-case flagship subject.",
        "evaluation": {"majority_vote_accuracy": float(r["majority_vote"]), "n_heldout_trials": int(r["n_trials"]), "seed": 0,
                       "onnx_vs_torch_max_abs_diff": max(err, err2)}})
    print(f"  exported {mid}: majority-vote {r['majority_vote']:.3f} on {r['n_trials']} held-out trials; ONNX parity {max(err, err2):.1e}")
    try:
        write_replay_log(task, sid, mode, model_name, r, order, mid, MANIFEST_ENTRIES[-1]["evaluation"])
    except Exception as exc:                                  # a replay-log problem must never lose the models
        print(f"  WARNING: replay log for {mid} was not written: {exc!r}")

## 7. Models: TCFormer (improved) and EEGNet, with per-model settings

In [ ]:
!pip install -q braindecode
from braindecode.models import TCFormer

def tcformer_factory_improved(n_channels, n_samples, n_classes):
    return TCFormer(n_chans=n_channels, n_times=n_samples, n_outputs=n_classes,
                     pool_length_1=4, pool_length_2=2)

_test_model = tcformer_factory_improved(64, 160, 3)
_tokens = {}
_test_model.to_tokens.register_forward_hook(lambda mod, inp, out: _tokens.update(n=out.shape[1]))
_test_out = _test_model(torch.randn(4, 64, 160))
print(f"Improved TCFormer sanity check: {sum(p.numel() for p in _test_model.parameters())} params, "
      f"output shape {tuple(_test_out.shape)}, {_tokens['n']} transformer tokens "
      f"(was 2 with the published default pooling - same param count either way)")

In [ ]:
MODEL_CFG = {
    "tcformer": dict(factory=tcformer_factory_improved,
                     kw=dict(optimizer_name="adamw", weight_decay=1e-2, label_smoothing=0.1, keep_frac=0.5,
                             use_warmup=True, warmup_frac=0.1, base_lr=5e-4, ft_lr=3e-4),
                     cross_base_lr=3e-4, base_epochs=250, ft_epochs=100, base_patience=25, ft_patience=18),
    "eegnet": dict(factory=eegnet_factory,
                   kw=dict(optimizer_name="adam", weight_decay=0.0, label_smoothing=0.0, keep_frac=0.3,
                           use_warmup=False, warmup_frac=0.1, base_lr=1e-3, ft_lr=5e-4),
                   cross_base_lr=5e-4, base_epochs=150, ft_epochs=60, base_patience=20, ft_patience=12),
}
if EPOCH_OVERRIDE:
    for _m in MODEL_CFG.values():
        _m.update(base_epochs=EPOCH_OVERRIDE, ft_epochs=EPOCH_OVERRIDE, base_patience=EPOCH_OVERRIDE, ft_patience=EPOCH_OVERRIDE)

@torch.no_grad()
def predict_proba(model, X, batch=64):
    model.eval(); out = []
    for i in range(0, len(X), batch):
        xb = torch.tensor(X[i:i + batch], dtype=torch.float32).to(DEVICE)
        out.append(torch.softmax(model(xb), dim=1).cpu().numpy())
    return np.concatenate(out)

def trial_vote_accuracy(pred, y, trial_ids, n_classes):
    true_l, pred_l = [], []
    for t in np.unique(trial_ids):
        m = trial_ids == t
        true_l.append(y[m][0]); pred_l.append(np.bincount(pred[m], minlength=n_classes).argmax())
    return float(np.mean(np.array(true_l) == np.array(pred_l))), len(true_l)

_e = eegnet_factory(64, 160, 3)
print("EEGNet sanity check:", sum(p.numel() for p in _e.parameters()), "params, output", tuple(_e(torch.randn(2, 64, 160)).shape))

## 8. Stage 0 — external (cross-subject) pretraining, once per task and model

Builds one pool of windows from `EXT_IDS` (never an evaluated subject) per task, cached in `/kaggle/working/`. Each
model is pretrained on it once; all target subjects of that task start from those weights. Loading the pool is the slow
part (about 1-2 minutes per subject on the first pass).

In [ ]:
def build_external_pool(task, ext_ids, stride):
    cfg = TASK_CFG[task]; order = cfg["order"]
    cache = f"/kaggle/working/ext_pool_{task}_{len(ext_ids)}_{stride}_{DATA_TAG}.npz"
    if os.path.exists(cache):
        z = np.load(cache)
        print(f"  loaded cached external pool: {len(z['y'])} windows"); return z["X"].astype(np.float32), z["y"], z["tid"]
    Xs, ys, ts, offset = [], [], [], 0
    for k, sid in enumerate(ext_ids):
        try:
            X, y, tid, modes = load_subjects([sid], cfg["runs"], cfg["labels"], exclude_bad=False)
        except Exception as e:
            print(f"  external S{sid:03d} skipped ({type(e).__name__}: {e})"); continue
        keep = np.arange(0, len(X), stride)
        Xs.append(X[keep].astype(np.float16)); ys.append(np.array([order.index(l) for l in y])[keep]); ts.append(tid[keep] + offset)
        offset += int(tid.max()) + 1
        print(f"  external subject {k + 1}/{len(ext_ids)} (S{sid:03d}): {len(keep)} windows")
    X, y, tid = np.concatenate(Xs), np.concatenate(ys), np.concatenate(ts)
    np.savez_compressed(cache, X=X, y=y, tid=tid)
    return X.astype(np.float32), y, tid

def pretrain_external(task, model_name, seed=0):
    cfg = TASK_CFG[task]; n_classes = len(cfg["order"]); mc = MODEL_CFG[model_name]; kw = mc["kw"]
    X, y, tid = build_external_pool(task, EXT_IDS, EXT_WINDOW_STRIDE)
    set_seed(seed)
    tr_m, va_m = split_by_trial(tid, y, 0.1, seed=0)
    assert not (set(tid[tr_m].tolist()) & set(tid[va_m].tolist())), "external train/validation share a trial"
    model = mc["factory"](X.shape[1], X.shape[2], n_classes).to(DEVICE)
    crit = nn.CrossEntropyLoss(weight=class_weights(y[tr_m], n_classes).to(DEVICE), label_smoothing=kw["label_smoothing"])
    opt = make_optimizer(model.parameters(), kw["optimizer_name"], lr=kw["base_lr"], weight_decay=kw["weight_decay"])
    loader = make_loader(X[tr_m], y[tr_m], batch_size=32)
    sched = None
    if kw["use_warmup"]:
        sched = WarmupCosineScheduler(opt, warmup_steps=max(1, int(len(loader) * 0.1 * EXT_EPOCHS)),
                                      total_steps=max(2, len(loader) * EXT_EPOCHS), base_lr=kw["base_lr"])
    print(f"  [{model_name}] pretraining on {tr_m.sum()} windows from {len(EXT_IDS)} subjects (validation: {va_m.sum()} windows, by trial)")
    model, hist = train_loop(model, loader, make_loader(X[va_m], y[va_m], batch_size=32, shuffle=False), opt, crit,
                             max_epochs=EXT_EPOCHS, patience=EXT_PATIENCE, verbose=True, scheduler=sched)
    print(f"  [{model_name}] external pretraining finished: {len(hist)} epochs, best validation accuracy {max(h['val_acc'] for h in hist):.3f}")
    del X, y, tid
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

def restore_from_inputs():
    """Copy cached external pools and pretrained weights from an attached earlier run (Add Input) so they are not recomputed."""
    import glob, shutil
    for pat in ("ext_pool_*.npz", "ext_state_*.pt"):
        for f in glob.glob(f"/kaggle/input/**/{pat}", recursive=True):
            dst = os.path.join("/kaggle/working", os.path.basename(f))
            if not os.path.exists(dst):
                shutil.copy(f, dst); print(f"  restored {os.path.basename(f)} from an attached earlier run")

EXT_STATE = {}
if "cross" in VARIANTS:
    restore_from_inputs()
    for task in RUN_TASKS:
        for model_name in MODELS_TO_RUN:
            sp = f"/kaggle/working/ext_state_{task}_{model_name}_{N_EXT}_{DATA_TAG}.pt"
            if os.path.exists(sp):
                EXT_STATE[(task, model_name)] = torch.load(sp, map_location="cpu")
                print(f"  loaded pretrained weights {os.path.basename(sp)}: pretraining skipped"); continue
            print(f"=== Stage 0: Task {task}, {model_name} ===")
            EXT_STATE[(task, model_name)] = pretrain_external(task, model_name)
            torch.save(EXT_STATE[(task, model_name)], sp)       # never lose this again
            print(f"  saved pretrained weights -> {sp}  ({(time.time() - T0) / 3600:.1f} h used so far)")
else:
    print("EXPORT_VARIANT = baseline: no cross-subject pretraining, Stage 0 skipped")


## 9. Run: model x variant x seed, always on the same held-out trials

For each subject the data is loaded once and the split is verified. Every model/variant/seed is then trained, the
softmax outputs of the seeds are averaged (`ensemble`), and — when both models ran — the two models' averaged
probabilities are averaged again (`arch_ensemble`). The held-out trials are asserted identical everywhere.

In [ ]:
results = {} if DRY_RUN else load_results_merged()
print(f"{len(results)} finished record(s) already available" if results else "starting with no earlier results")
STOP_EARLY = False

def _out_of_time():
    return (not DRY_RUN) and time_left() < MIN_BLOCK_MINUTES * 60

def _keys_for(task, sid):
    keys = [f"{task}|{m}|{v}|{sid}" for m in MODELS_TO_RUN for v in VARIANTS]
    if len(MODELS_TO_RUN) > 1:
        keys += [f"{task}|arch_ensemble|{v}|{sid}" for v in VARIANTS]
    return keys


try:
    for task in RUN_TASKS:
        cfg = TASK_CFG[task]; order = cfg["order"]; K = len(order)
        print(f"\n=== Task {task} ===")
        for sid in (EXPORT_SUBJECTS[task] if EXPORT_MODE else EVAL_SUBJECTS):
            if all(k in results for k in _keys_for(task, sid)):
                print(f"  S{sid:03d}: already done, skipping"); continue
            if _out_of_time():
                print(f"  stopping before S{sid:03d}: only {time_left() / 60:.0f} min of the {MAX_RUNTIME_HOURS} h budget left"); STOP_EARLY = True; break
            X, y, tid, modes = load_subjects([sid], cfg["runs"], cfg["labels"], exclude_bad=False)
            y_int = np.array([order.index(l) for l in y])
            n_eval = verify_no_leakage(y_int, tid, modes, f"Task {task} S{sid:03d}")
            reference_eval, sub_probs = {}, {}
            for model_name in MODELS_TO_RUN:
                mc = MODEL_CFG[model_name]
                for v in VARIANTS:
                    probs, per_seed = {"ME": [], "MI": []}, {"ME": [], "MI": []}
                    for seed in SEEDS[model_name]:
                        kw = dict(mc["kw"])
                        if v == "cross":
                            kw["base_lr"] = mc["cross_base_lr"]
                        _, res_by_mode, _ = run_finetune_protocol_hybrid(
                            X, y_int, tid, modes, order, mc["factory"],
                            base_epochs=mc["base_epochs"], ft_epochs=mc["ft_epochs"],
                            base_patience=mc["base_patience"], ft_patience=mc["ft_patience"], verbose=False,
                            init_state=(EXT_STATE[(task, model_name)] if v == "cross" else None), seed=seed, **kw)
                        for m, r in res_by_mode.items():
                            if EXPORT_MODE and model_name in EXPORT_MODELS and v == EXPORT_VARIANT and seed == SEEDS[model_name][0]:
                                export_entry(task, sid, m, model_name, r, order, X.shape[1], X.shape[2])
                            Xe, ye, te = r["eval_data"]
                            if m in reference_eval:
                                assert np.array_equal(reference_eval[m][0], ye) and np.array_equal(reference_eval[m][1], te), \
                                    "held-out trials differ between runs"
                            else:
                                reference_eval[m] = (ye, te)
                            p = predict_proba(r["model"], Xe)
                            check_probs(p, len(Xe), K, f"S{sid:03d} {model_name} {v} seed {seed} {m}")
                            probs[m].append(p); per_seed[m].append(float(r["majority_vote"]))
                        del res_by_mode
                    rec, line = {}, f"  S{sid:03d} {model_name:8s} {v:8s}"
                    sub_probs[(model_name, v)] = {}
                    for m in ("ME", "MI"):
                        if probs[m]:
                            ye, te = reference_eval[m]
                            avg = np.mean(probs[m], axis=0); sub_probs[(model_name, v)][m] = avg
                            ens, n = trial_vote_accuracy(avg.argmax(1), ye, te, K)
                            rec[m] = {"per_seed": per_seed[m], "ensemble": ens, "n_trials": n}
                            line += f" | {m}: seeds {np.round(per_seed[m], 3).tolist()} -> ens {ens:.3f} ({n} trials)"
                    results[f"{task}|{model_name}|{v}|{sid}"] = rec
                    print(line)
            if len(MODELS_TO_RUN) > 1:
                for v in VARIANTS:
                    rec, line = {}, f"  S{sid:03d} arch_ens {v:8s}"
                    for m in ("ME", "MI"):
                        ps = [sub_probs[(mn, v)][m] for mn in MODELS_TO_RUN if m in sub_probs.get((mn, v), {})]
                        if len(ps) == len(MODELS_TO_RUN):
                            ye, te = reference_eval[m]
                            ens, n = trial_vote_accuracy(np.mean(ps, axis=0).argmax(1), ye, te, K)
                            rec[m] = {"per_seed": [], "ensemble": ens, "n_trials": n}
                            line += f" | {m}: {ens:.3f} ({n} trials)"
                    results[f"{task}|arch_ensemble|{v}|{sid}"] = rec
                    print(line)
            np.savez_compressed(os.path.join(PROBS_DIR, f"task{task}_S{sid:03d}.npz"),
                                **{f"{mn}|{v}|{m}": a for (mn, v), d in sub_probs.items() for m, a in d.items()},
                                **{f"y|{m}": reference_eval[m][0] for m in reference_eval},
                                **{f"trial|{m}": reference_eval[m][1] for m in reference_eval})
            save_results(results)                       # after every subject (dry runs write the _DRYRUN file)
            print(f"  [saved {RESULTS_PATH}: {len(results)} record(s)]")
            del X, y, tid, modes
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        if STOP_EARLY: break
finally:
    save_results(results)          # also runs if the cell crashes: whatever finished is on disk

size = os.path.getsize(RESULTS_PATH) if os.path.exists(RESULTS_PATH) else 0
print(f"\nresults file: {RESULTS_PATH} | {size} bytes | {len(results)} record(s)" + ("  (DRY RUN file)" if DRY_RUN else ""))
assert size > 2 or STOP_EARLY, f"results file {RESULTS_PATH} is missing or empty"
if STOP_EARLY:
    print("Stopped early to stay inside the session limit. Download the JSON, then start another version with "
          "PRIOR_RESULTS_PATHS pointing at it to continue.")

## 10. Summary

**Pre-specified primary outcome:** seed-averaged **TCFormer with cross-subject pretraining**
(`tcformer | cross`) — chosen before the full run because it is the variant the pilot supported. Everything else
(EEGNet, the architecture ensemble, baselines) is reported as secondary. Do **not** swap the primary after looking at
the held-out results; that would be selection on the test set.

In [ ]:
from scipy import stats

def _present(task):
    pairs = {(k.split("|")[1], k.split("|")[2]) for k in results if k.startswith(f"{task}|")}
    mo, vo = {"tcformer": 0, "eegnet": 1, "arch_ensemble": 2}, {"baseline": 0, "cross": 1}
    return sorted(pairs, key=lambda t: (mo.get(t[0], 9), vo.get(t[1], 9)))

def _min_correct_above_chance(n_trials, k, alpha=0.05):
    """Fewest correct trials that beat chance (1/k) by a one-sided exact binomial test."""
    for c in range(n_trials + 1):
        if 1 - stats.binom.cdf(c - 1, n_trials, 1.0 / k) < alpha:
            return c
    return n_trials + 1

def _paired(table, a, b, label):
    common = sorted(set(table[a]) & set(table[b]))
    if len(common) < 3:
        return
    xa = np.array([table[a][s]["ensemble"] for s in common]); xb = np.array([table[b][s]["ensemble"] for s in common]); d = xa - xb
    p_t = stats.ttest_rel(xa, xb).pvalue
    try:
        p_w = stats.wilcoxon(d).pvalue if np.any(d != 0) else 1.0
    except ValueError:
        p_w = float("nan")
    print(f"  {label}: {d.mean() * 100:+.1f} pts (sd of differences {d.std(ddof=1) * 100:.1f}); better/tie/worse = "
          f"{int((d > 0).sum())}/{int((d == 0).sum())}/{int((d < 0).sum())} of {len(common)}; paired t p={p_t:.3f}, Wilcoxon p={p_w:.3f}")

def summarize_v5(results):
    tasks = sorted({k.split("|")[0] for k in results})
    for task in tasks:
        K = len(TASK_CFG[task]["order"])
        subs = sorted({int(k.split("|")[3]) for k in results if k.startswith(f"{task}|")})
        for mode in ("ME", "MI"):
            print(f"\n--- Task {task} {mode} (chance {100 / K:.0f}%) ---")
            table = {}
            for (m, v) in _present(task):
                recs = {sid: results[f"{task}|{m}|{v}|{sid}"][mode] for sid in subs
                        if f"{task}|{m}|{v}|{sid}" in results and mode in results[f"{task}|{m}|{v}|{sid}"]}
                if not recs: continue
                table[(m, v)] = recs
                ens = np.array([r["ensemble"] for r in recs.values()]); n = len(ens)
                single = np.array([np.mean(r["per_seed"]) for r in recs.values()]) if m != "arch_ensemble" else None
                ci = stats.t.ppf(0.975, n - 1) * ens.std(ddof=1) / np.sqrt(n) if n > 1 else float("nan")
                thr = _min_correct_above_chance(int(round(np.mean([r["n_trials"] for r in recs.values()]))), K)
                n_sig = sum(int(round(r["ensemble"] * r["n_trials"])) >= thr for r in recs.values())
                tag = "  <-- PRIMARY" if (m, v) == ("tcformer", "cross") else ""
                print(f"  {m:13s} {v:8s} n={n:2d} | seed-ensemble mean {ens.mean() * 100:5.1f}% (std {ens.std() * 100:4.1f} [population, as in the paper eq. 15], 95% CI +/-{ci * 100:4.1f})"
                      + (f" | mean single-seed {single.mean() * 100:5.1f}%" if single is not None else "")
                      + f" | above chance (exact binomial, p<0.05): {n_sig}/{n}" + tag)
            print("  -- paired comparisons on the same held-out trials (differences under ~3-4 points are noise) --")
            for m in ("tcformer", "eegnet"):
                if (m, "cross") in table and (m, "baseline") in table:
                    _paired(table, (m, "cross"), (m, "baseline"), f"{m}: cross - baseline")
            for v in ("cross", "baseline"):
                if ("tcformer", v) in table and ("eegnet", v) in table:
                    _paired(table, ("tcformer", v), ("eegnet", v), f"tcformer - eegnet ({v})")
                if ("arch_ensemble", v) in table and ("tcformer", v) in table:
                    _paired(table, ("arch_ensemble", v), ("tcformer", v), f"arch_ensemble - tcformer ({v})")
    print("\nSmall-sample numbers (~18 held-out trials per subject, each trial = 5.6 points). Report the primary outcome as "
          "mean +/- std across subjects, state the number of seeds averaged, and quote a difference only with its paired test.")

summarize_v5(results)

## 11. After the run

1. Download `v5_results.json` and the `probs/` folder (one file per subject). Bring them back and I will update the
   paper's tables, including the CNN-versus-transformer comparison under the final trial-level split.
2. Suggested sessions: (a) `RUN_TASKS = ["B"]`, both models, `cross` (about 8-9 h); (b) `ABLATION_BASELINE = True`, `RUN_TASKS = ["A"]`
   (TCFormer without cross-subject pretraining, about 5 h) so the paper can state the cross-subject gain from a paired comparison;
   (c) only if you want to test the preprocessing hypotheses: `NORM_MODE = "run"` and/or `CHANNEL_SUBSET = SENSORIMOTOR_CHANNELS`
   with `RUN_TASKS = ["A"]`, TCFormer `cross` only (they write separate result files).
   Merge sessions by listing earlier files in `PRIOR_RESULTS_PATHS`; keys already contain task, model, variant and subject.
3. Quote the **primary outcome** (TCFormer + cross-subject pretraining, seed-averaged) with mean +/- std across
   subjects; report the architecture ensemble and EEGNet as secondary results.
4. Checkpoint / UI export is not part of v5; once the recipe is settled, use v3.1 for exports.

In [ ]:
# EXPORT: write the manifest fragment and zip everything for download
if not MANIFEST_ENTRIES:
    print("NOTHING WAS EXPORTED. Look above for 'stopping before' or an error message. Nothing is lost that was saved earlier:")
    print("cached pools (ext_pool_*.npz) and pretrained weights (ext_state_*.pt) are in /kaggle/working and can be attached to the next run.")
else:
    with open(os.path.join(EXPORT_DIR, "manifest_entries.json"), "w") as fh:
        json.dump({"models": MANIFEST_ENTRIES}, fh, indent=1)
    zip_path = "/kaggle/working/neurobot_models.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for f in sorted(os.listdir(EXPORT_DIR)):
            z.write(os.path.join(EXPORT_DIR, f), f)
    print(f"{len(MANIFEST_ENTRIES)} model(s) exported -> {zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB)")
    for e in MANIFEST_ENTRIES:
        print(" ", e["id"], "| variant", e.get("variant"), "| held-out majority vote", round(e["evaluation"]["majority_vote_accuracy"], 3), "on", e["evaluation"]["n_heldout_trials"], "trials")


## After the run

1. Download **`neurobot_models.zip`** from the **Output** tab of this version (it holds the ONNX files, the replay logs and `manifest_entries.json`).
2. Unzip it, copy the `.onnx` files into `backend/checkpoints/` and **merge `manifest_entries.json` into `backend/checkpoints/manifest.json`** (append its entries to the `models` list). Copy the `replay_*.json` files into `backend/data/`.
3. Run `pytest backend/tests` and start the backend; `/api/health` must list the new models as available.
4. Commit and push: Render redeploys automatically.

**Using the expensive `"cross"` variant later (optional):** run one task per session (`RUN_TASKS` is set to `["B"]`, change to `["A"]` for the second). After the first
session, create a Kaggle dataset from that version's output (*Output -> New Dataset*) and attach it to the next session: cached pools and pretrained weights are restored automatically.
The paper reports whichever variant is actually deployed (the manifest says which).
